# **Lab Day7**

___

## Installations

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
def make_df(cols, ind)->pd.DataFrame: 
    """Quickly create a DataFrame.""" 
    data = {}
    for col in cols: 
        values = [] 
        for i in ind: 
            values.append(str(col) + str(i)) 
        data[col] = values 
    return pd.DataFrame(data, index=ind)

make_df('ABC', range(3))

,A,B,C
0,A0,B0,C0
1,A1,B1,C1
2,A2,B2,C2


In [3]:
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                        for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                            for a in self.args)

## Aggregation and Grouping

### Aggregations in Pandas

- **Aggregation** is the process of summarizing a large dataset using a **single value** or a smaller set of values.
- Common aggregation functions include:
  - **`sum()`** → Calculates the total.
  - **`mean()`** → Calculates the average.
  - **`median()`** → Finds the middle value.
  - **`min()`** → Finds the smallest value.
  - **`max()`** → Finds the largest value.
- Pandas provides aggregation functions similar to those available in **NumPy**.
- Pandas also provides more advanced aggregation using **`groupby()`**.

#### Main Idea

> Aggregation helps us **summarize and understand large datasets** using useful statistics.

In [4]:
planets = sns.load_dataset('planets')
planets.shape

(1035, 6)

In [5]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [6]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
ser

0    0.374540
1    0.950714
2    0.731994
3    0.598658
4    0.156019
dtype: float64

In [7]:
ser.sum()

np.float64(2.811925491708157)

In [8]:
ser.mean()

np.float64(0.5623850983416314)

In [9]:
df = pd.DataFrame({'A': rng.rand(5),
                    'B': rng.rand(5)})
df

,A,B
0,0.155995,0.020584
1,0.058084,0.969910
2,0.866176,0.832443
3,0.601115,0.212339
4,0.708073,0.181825


In [10]:
df.mean()

A    0.477888
B    0.443420
dtype: float64

By specifying the `axis` argument, you can instead aggregate within each row:

In [11]:
df.mean(axis=1)

0    0.088290
1    0.513997
2    0.849309
3    0.406727
4    0.444949
dtype: float64

In [12]:
planets.describe()

,number,orbital_period,mass,distance,year
count,1035.000000,992.000000,513.000000,808.000000,1035.000000
mean,1.785507,2002.917596,2.638161,264.069282,2009.070531
std,1.240976,26014.728304,3.818617,733.116493,3.972567
min,1.000000,0.090706,0.003600,1.350000,1989.000000
25%,1.000000,5.442540,0.229000,32.560000,2007.000000
50%,1.000000,39.979500,1.260000,55.250000,2010.000000
75%,2.000000,526.005000,3.040000,178.500000,2012.000000
max,7.000000,730000.000000,25.000000,8500.000000,2014.000000


In [13]:
planets.dropna().describe()

,number,orbital_period,mass,distance,year
count,498.00000,498.000000,498.000000,498.000000,498.000000
mean,1.73494,835.778671,2.509320,52.068213,2007.377510
std,1.17572,1469.128259,3.636274,46.596041,4.167284
min,1.00000,1.328300,0.003600,1.350000,1989.000000
25%,1.00000,38.272250,0.212500,24.497500,2005.000000
50%,1.00000,357.000000,1.245000,39.940000,2009.000000
75%,2.00000,999.600000,2.867500,59.332500,2011.000000
max,6.00000,17337.500000,25.000000,354.000000,2014.000000


The following table summarizes some other built-in Pandas aggregations:

| Aggregation              | Returns                         |
|--------------------------|---------------------------------|
| ``count``                | Total number of items           |
| ``first``, ``last``      | First and last item             |
| ``mean``, ``median``     | Mean and median                 |
| ``min``, ``max``         | Minimum and maximum             |
| ``std``, ``var``         | Standard deviation and variance |
| ``mad``                  | Mean absolute deviation         |
| ``prod``                 | Product of all items            |
| ``sum``                  | Sum of all items                |

These are all methods of `DataFrame` and `Series` objects.

### groupby: Split, Apply, Combine

Simple aggregations can give you a flavor of your dataset, but often we would prefer to aggregate conditionally on some label or index: this is implemented in the so-called `groupby` operation.
The name "group by" comes from a command in the SQL database language, but it is perhaps more illuminative to think of it in the terms first coined by Hadley Wickham of Rstats fame: *split, apply, combine*.

A canonical example of this split-apply-combine operation, where the "apply" is a summation aggregation, is illustrated in this figure:

![](split-apply-combine.png)

([figure source in Appendix](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/06.00-Figure-Code.ipynb#Split-Apply-Combine))

This illustrates what the `groupby` operation accomplishes:

- The *split* step involves breaking up and grouping a `DataFrame` depending on the value of the specified key.
- The *apply* step involves computing some function, usually an aggregate, transformation, or filtering, within the individual groups.
- The *combine* step merges the results of these operations into an output array.

While this could certainly be done manually using some combination of the masking, aggregation, and merging commands covered earlier, an important realization is that *the intermediate splits do not need to be explicitly instantiated*. Rather, the `groupby` can (often) do this in a single pass over the data, updating the sum, mean, count, min, or other aggregate for each group along the way.
The power of the `groupby` is that it abstracts away these steps: the user need not think about *how* the computation is done under the hood, but rather can think about the *operation as a whole*.

As a concrete example, let's take a look at using Pandas for the computation shown in the following figure.
We'll start by creating the input `DataFrame`:

In [14]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                    'data': range(6)})
df

,key,data
0,A,0
1,B,1
2,C,2
3,A,3
4,B,4
5,C,5


In [15]:
df.groupby('key')

In [16]:
df.groupby('key').sum()

,data
key,
A,3
B,5
C,7


In [17]:
planets.head()


,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [18]:
planets.groupby('method')

In [19]:
planets.groupby('method')['orbital_period']

In [20]:
planets.groupby('method')['orbital_period'].median()

method
Astrometry                         631.180000
Eclipse Timing Variations         4343.500000
Imaging                          27500.000000
Microlensing                      3300.000000
Orbital Brightness Modulation        0.342887
Pulsar Timing                       66.541900
Pulsation Timing Variations       1170.000000
Radial Velocity                    360.200000
Transit                              5.714932
Transit Timing Variations           57.011000
Name: orbital_period, dtype: float64

In [21]:
for (method, group) in planets.groupby('method'):
    print("method: {0:30s} shape={1}".format(method, group.shape))

method: Astrometry                     shape=(2, 6)
method: Eclipse Timing Variations      shape=(9, 6)
method: Imaging                        shape=(38, 6)
method: Microlensing                   shape=(23, 6)
method: Orbital Brightness Modulation  shape=(3, 6)
method: Pulsar Timing                  shape=(5, 6)
method: Pulsation Timing Variations    shape=(1, 6)
method: Radial Velocity                shape=(553, 6)
method: Transit                        shape=(397, 6)
method: Transit Timing Variations      shape=(4, 6)


### Dispatch methods

In [22]:
planets['year'].describe()

count    1035.000000
mean     2009.070531
std         3.972567
min      1989.000000
25%      2007.000000
50%      2010.000000
75%      2012.000000
max      2014.000000
Name: year, dtype: float64

In [23]:
planets.groupby('method')['year'].describe()

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
Astrometry,2.0,2011.500000,2.121320,2010.0,2010.75,2011.5,2012.25,2013.0
Eclipse Timing Variations,9.0,2010.000000,1.414214,2008.0,2009.00,2010.0,2011.00,2012.0
Imaging,38.0,2009.131579,2.781901,2004.0,2008.00,2009.0,2011.00,2013.0
Microlensing,23.0,2009.782609,2.859697,2004.0,2008.00,2010.0,2012.00,2013.0
Orbital Brightness Modulation,3.0,2011.666667,1.154701,2011.0,2011.00,2011.0,2012.00,2013.0
Pulsar Timing,5.0,1998.400000,8.384510,1992.0,1992.00,1994.0,2003.00,2011.0
Pulsation Timing Variations,1.0,2007.000000,NaN,2007.0,2007.00,2007.0,2007.00,2007.0
Radial Velocity,553.0,2007.518987,4.249052,1989.0,2005.00,2009.0,2011.00,2014.0
Transit,397.0,2011.236776,2.077867,2002.0,2010.00,2012.0,2013.00,2014.0


### Aggregate, Filter, Transform, Apply

GroupBy Operations

- `groupby()` provides several operations for working with grouped data.
- In addition to **aggregation**, `GroupBy` objects support:
  - **`aggregate()`** → Apply one or more aggregation functions to each group.
  - **`filter()`** → Keep or remove groups based on a condition.
  - **`transform()`** → Perform calculations on each group and return results with the original shape.
  - **`apply()`** → Apply a custom function to each group.
- These operations allow us to perform different calculations **before combining the grouped data**.

Main Idea

> `GroupBy` provides several methods to efficiently process and analyze data **group by group**.

Common `GroupBy` Methods

| Method | Purpose |
|---|---|
| `aggregate()` | Calculate summary statistics for each group |
| `filter()` | Filter groups based on a condition |
| `transform()` | Transform values while keeping the original shape |
| `apply()` | Apply a custom function to each group |

- The following examples will use a specific `DataFrame` to demonstrate these operations.

In [24]:
rng = np.random.RandomState(0)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                    'data1': range(6),
                    'data2': rng.randint(0, 10, 6)},
                    columns = ['key', 'data1', 'data2'])
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


#### aggregate

In [25]:
df.groupby('key').aggregate(['min', np.median, 'max'])

data1            data2           
      min median max   min median max
key                                  
A       0    1.5   3     3    4.0   5
B       1    2.5   4     0    3.5   7
C       2    3.5   5     3    6.0   9

In [26]:
df.groupby('key').aggregate({'data1': 'min',
                            'data2': 'max'})

,data1,data2
key,,
A,0,5
B,1,7
C,2,9


#### Filtering

In [27]:
def filter_func(x):
    return x['data2'].std() > 4

display('df', "df.groupby('key').std()",
        "df.groupby('key').filter(filter_func)")

df
  key  data1  data2
0   A      0      5
1   B      1      0
2   C      2      3
3   A      3      3
4   B      4      7
5   C      5      9

df.groupby('key').std()
       data1     data2
key                   
A    2.12132  1.414214
B    2.12132  4.949747
C    2.12132  4.242641

df.groupby('key').filter(filter_func)
  key  data1  data2
1   B      1      0
2   C      2      3
4   B      4      7
5   C      5      9

#### Transformation

In [28]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [29]:
def center(x:pd.DataFrame):
    return x - x.mean(axis=0)
df.groupby('key').transform(center)

,data1,data2
0,-1.5,1.0
1,-1.5,-3.5
2,-1.5,-3.0
3,1.5,-1.0
4,1.5,3.5
5,1.5,3.0


#### apply

In [30]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [31]:
def norm_by_data2(x:pd.DataFrame):
    # x is a DataFrame of group values
    x['data1'] /= x['data2'].sum()
    return x

df.groupby('key').apply(norm_by_data2)

data1  data2
key                   
A   0  0.000000      5
    3  0.375000      3
B   1  0.142857      0
    4  0.571429      7
C   2  0.166667      3
    5  0.416667      9

### Specifying the Split Key

Group Specification in `groupby()`

- In simple examples, we used **one column** to define the groups.
- However, `groupby()` provides several ways to define groups.
- Groups can be created using:
  - A single column.
  - Multiple columns.
  - Other grouping criteria or functions.
- The way groups are defined determines **how the data is split** before applying an operation.

Main Idea

> `groupby()` is flexible and allows you to define groups in different ways depending on the analysis you need.

In [32]:
df

,key,data1,data2
0,A,0,5
1,B,1,0
2,C,2,3
3,A,3,3
4,B,4,7
5,C,5,9


In [33]:
L = [0, 1, 0, 1, 2, 0] 
df.groupby(L).sum()

,key,data1,data2
0,ACC,7,17
1,BA,4,3
2,B,4,7


In [34]:
df.groupby('key').sum()

,data1,data2
key,,
A,3,8
B,5,7
C,7,12


### A dictionary or series mapping index to group

Another method is to provide a dictionary that maps index values to the group keys:

In [35]:
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
display('df2', 'df2.groupby(mapping).sum()')

,data1,data2
key,,
A,0,5
B,1,0
C,2,3
A,3,3
B,4,7
C,5,9
,data1,data2
key,,
consonant,12,19


In [36]:
df2.groupby(str.lower).mean()

,data1,data2
key,,
a,1.5,4.0
b,2.5,3.5
c,3.5,6.0


### A list of valid keys

Further, any of the preceding key choices can be combined to group on a multi-index:

In [37]:
df2.groupby([str.lower, mapping]).mean()

,,data1,data2
key,key,,
a,vowel,1.5,4.0
b,consonant,2.5,3.5
c,consonant,3.5,6.0


### Grouping Example

As an example of this, in a few lines of Python code we can put all these together and count discovered planets by method and by decade:

In [38]:
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'
planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)

decade,1980s,1990s,2000s,2010s
method,,,,
Astrometry,0.0,0.0,0.0,2.0
Eclipse Timing Variations,0.0,0.0,5.0,10.0
Imaging,0.0,0.0,29.0,21.0
Microlensing,0.0,0.0,12.0,15.0
Orbital Brightness Modulation,0.0,0.0,0.0,5.0
Pulsar Timing,0.0,9.0,1.0,1.0
Pulsation Timing Variations,0.0,0.0,1.0,0.0
Radial Velocity,1.0,52.0,475.0,424.0
Transit,0.0,0.0,64.0,712.0


This shows the power of combining many of the operations we've discussed up to this point when looking at realistic datasets: we quickly gain a coarse understanding of when and how extrasolar planets were detected in the years after the first discovery.

I would suggest digging into these few lines of code and evaluating the individual steps to make sure you understand exactly what they are doing to the result.
It's certainly a somewhat complicated example, but understanding these pieces will give you the means to similarly explore your own data.

## Pivot

Pivot Tables

- A **pivot table** is a way to summarize and analyze data in a **two-dimensional table**.
- Pivot tables are commonly used in:
  - Spreadsheets
  - Tabular data analysis
  - Data summarization
- A pivot table takes **column-based data** as input and organizes it into a structured table.
- It allows us to analyze data across **multiple dimensions**.

Pivot Table vs `groupby()`

- `groupby()` is commonly used to group data based on **one or more columns** and then perform an aggregation.
- A pivot table can be thought of as a **multidimensional version of `groupby()`**.
- Both use the **split-apply-combine** approach:
  - **Split** → Divide the data into groups.
  - **Apply** → Perform an operation or aggregation.
  - **Combine** → Combine the results into a summary table.

Main Difference

- **`groupby()`** → Mainly organizes the result along an index/group structure.
- **Pivot table** → Organizes the result into a **two-dimensional grid** using rows and columns.

Main Idea

> **Pivot table = Multidimensional `groupby()` aggregation**

```text
groupby()
    ↓
Split → Apply → Combine
    ↓
One-dimensional grouping

pivot_table()
    ↓
Split → Apply → Combine
    ↓
Two-dimensional grouping

In [39]:
import seaborn as sns
titanic = sns.load_dataset('titanic')

In [40]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [41]:
titanic.shape

(891, 15)

### Pivot Tables by Hand

- To explore the data, we can group the records based on:
  - **Sex**
  - **Survival status**
  - A combination of both
- Using `groupby()`, we can calculate the **survival rate by sex**.

In [42]:
titanic.groupby('sex')[['survived']].mean()

,survived
sex,
female,0.742038
male,0.188908


Survival Rate by Sex and Class

- The previous analysis showed that **survival rate differs between females and males**.
- We can go one step further by analyzing survival based on **both sex and class**.
- Using `groupby()`, we can follow the **split-apply-combine** process.

Steps

1. **Group by** `class` and `sex`.
2. **Select** the `survived` column.
3. **Apply** the `mean()` aggregation.
4. **Combine** the results.
5. **Unstack** the hierarchical index to create a two-dimensional table.

In [43]:
titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()

class,First,Second,Third
sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


### Pivot Table Syntax

Using `pivot_table()`

- The previous `groupby()` approach gives us useful information about how **sex** and **class** affected survival.
- However, the code can become **long and difficult to read**.

In [44]:
df.iloc[1,1]

np.int64(1)

In [45]:
titanic.pivot_table(values='survived', index='sex', columns='class', aggfunc='mean',
                    margins=True, margins_name='Total SRate', dropna=True)* 100

class,First,Second,Third,Total SRate
sex,,,,
female,96.808511,92.105263,50.000000,74.203822
male,36.885246,15.740741,13.544669,18.890815
Total SRate,62.962963,47.282609,24.236253,38.383838


### Multilevel Pivot Tables

- Like `groupby()`, a **pivot table** can group data using **multiple levels**.
- We can add additional dimensions to make the analysis more detailed.
- In this example, we want to analyze survival based on:
  - **Sex**
  - **Class**
  - **Age**
- Since `age` contains many different values, we can divide the ages into **ranges (bins)**.
- Pandas provides **`pd.cut()`** to divide continuous numerical values into categories.

In [46]:
from pandas import col


age = pd.cut(titanic['age'], [0, 18, 80]) # 0 < age ≤ 18 , 18 < age ≤ 80
titanic.pivot_table(values='survived', index=['sex', age], columns='class',aggfunc='sum')

class            First  Second  Third
sex    age                           
female (0, 18]      10      14     22
       (18, 80]     72      54     25
male   (0, 18]       4       9     11
       (18, 80]     36       6     27

In [47]:
fare = pd.qcut(titanic['fare'], 2)
fare

0       (-0.001, 14.454]
1      (14.454, 512.329]
2       (-0.001, 14.454]
3      (14.454, 512.329]
4       (-0.001, 14.454]
             ...        
886     (-0.001, 14.454]
887    (14.454, 512.329]
888    (14.454, 512.329]
889    (14.454, 512.329]
890     (-0.001, 14.454]
Name: fare, Length: 891, dtype: category
Categories (2, interval[float64, right]): [(-0.001, 14.454] < (14.454, 512.329]]

In [48]:
titanic.pivot_table(values='survived', index=['sex', age], columns=[fare, 'class'],aggfunc='sum')

fare            (-0.001, 14.454]              (14.454, 512.329]             
class                      First Second Third             First Second Third
sex    age                                                                  
female (0, 18]               NaN    3.0  15.0              10.0   11.0   7.0
       (18, 80]              NaN   22.0  16.0              72.0   32.0   9.0
male   (0, 18]               NaN    0.0   6.0               4.0    9.0   5.0
       (18, 80]              0.0    5.0  22.0              36.0    1.0   5.0

### Additional Pivot Table Options

`DataFrame.pivot_table()` has several options that control how the pivot table is created.

Syntax

```python
DataFrame.pivot_table(
    data,
    values=None,
    index=None,
    columns=None,
    aggfunc='mean',
    fill_value=None,
    margins=False,
    dropna=True,
    margins_name='All',
    observed=False,
    sort=True
)

Important `pivot_table()` Parameters

- **`values`** → Specifies the column(s) whose values will be summarized.
- **`index`** → Specifies the column(s) used as the **rows** of the pivot table.
- **`columns`** → Specifies the column(s) used as the **columns** of the pivot table.
- **`aggfunc`** → Specifies the **aggregation function** such as `mean`, `sum`, `count`, `min`, or `max`.
- **`fill_value`** → Replaces missing values in the pivot table with a specified value.
- **`dropna`** → Controls whether rows or columns containing missing values are included.
- **`margins`** → Adds **totals** for rows and columns.
- **`margins_name`** → Specifies the name of the totals row and column. Default is `"All"`.
- **`observed`** → Controls which combinations of categorical variables are displayed.
- **`sort`** → Controls whether the result is sorted.

In [49]:
titanic.pivot_table(values='survived', index='sex', columns='class')

class,First,Second,Third
sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [50]:
titanic.pivot_table(values='survived', index='sex', columns='class',aggfunc='sum')

class,First,Second,Third
sex,,,
female,91,70,72
male,45,17,47


In [51]:
titanic.pivot_table(index='sex', columns='class',aggfunc={'survived':'sum', 'fare':'mean'})

fare                       survived             
class        First     Second      Third    First Second Third
sex                                                           
female  106.125798  21.970121  16.118810       91     70    72
male     67.226127  19.741782  12.661633       45     17    47

Notice also here that we've omitted the `values` keyword; when specifying a mapping for `aggfunc`, this is determined automatically

In [52]:
age = pd.cut(titanic['age'], [0, 18, 80])
fare = pd.qcut(titanic['fare'], 2)
titanic.pivot_table('survived', ['sex', age], [fare, 'class'],aggfunc='sum')

fare            (-0.001, 14.454]              (14.454, 512.329]             
class                      First Second Third             First Second Third
sex    age                                                                  
female (0, 18]               NaN    3.0  15.0              10.0   11.0   7.0
       (18, 80]              NaN   22.0  16.0              72.0   32.0   9.0
male   (0, 18]               NaN    0.0   6.0               4.0    9.0   5.0
       (18, 80]              0.0    5.0  22.0              36.0    1.0   5.0

In [53]:
titanic.pivot_table('survived', ['sex', age], [fare, 'class'],aggfunc='sum')

fare            (-0.001, 14.454]              (14.454, 512.329]             
class                      First Second Third             First Second Third
sex    age                                                                  
female (0, 18]               NaN    3.0  15.0              10.0   11.0   7.0
       (18, 80]              NaN   22.0  16.0              72.0   32.0   9.0
male   (0, 18]               NaN    0.0   6.0               4.0    9.0   5.0
       (18, 80]              0.0    5.0  22.0              36.0    1.0   5.0

In [54]:
titanic.pivot_table(values='survived', index='sex', columns='class',aggfunc='sum',margins=True)

class,First,Second,Third,All
sex,,,,
female,91,70,72,233
male,45,17,47,109
All,136,87,119,342


In [55]:
titanic.pivot_table(values='survived', index='sex', columns='class',aggfunc='sum',margins=True,margins_name='sum')

class,First,Second,Third,sum
sex,,,,
female,91,70,72,233
male,45,17,47,109
sum,136,87,119,342


In [56]:
titanic.pivot_table(values='survived', index='sex', columns='class',aggfunc='sum',margins=True,margins_name='sum',sort=True)

class,First,Second,Third,sum
sex,,,,
female,91,70,72,233
male,45,17,47,109
sum,136,87,119,342


## Tasks

### 1. Load and inspect the data

In [57]:
births = pd.read_csv("data/births.csv")
births.head()

,year,month,day,gender,births
0,1969,1,1.0,F,4046
1,1969,1,1.0,M,4440
2,1969,1,2.0,F,4454
3,1969,1,2.0,M,4548
4,1969,1,3.0,F,4548


In [58]:
births.info()
births.describe(include="all")


<class 'pandas.DataFrame'>
RangeIndex: 15547 entries, 0 to 15546
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    15547 non-null  int64  
 1   month   15547 non-null  int64  
 2   day     15067 non-null  float64
 3   gender  15547 non-null  str    
 4   births  15547 non-null  int64  
dtypes: float64(1), int64(3), str(1)
memory usage: 607.4 KB


,year,month,day,gender,births
count,15547.000000,15547.000000,15067.000000,15547,15547.000000
unique,NaN,NaN,NaN,2,NaN
top,NaN,NaN,NaN,F,NaN
freq,NaN,NaN,NaN,7776,NaN
mean,1979.037435,6.515919,17.769894,NaN,9762.293561
std,6.728340,3.449632,15.284034,NaN,28552.465810
min,1969.000000,1.000000,1.000000,NaN,1.000000
25%,1974.000000,4.000000,8.000000,NaN,4358.000000
50%,1979.000000,7.000000,16.000000,NaN,4814.000000
75%,1984.000000,10.000000,24.000000,NaN,5289.500000


### Task 1 — Year × Gender Analysis

Create a pivot table showing the **total number of births for each year and gender**.

Requirements:
- `year` must be the index.
- `gender` must become the columns.
- Aggregate `births` using `sum`.
- Add row and column totals using `margins=True`.


In [ ]:
births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).head()

gender,F,M,Total
year,,,
1969,1753634,1846572,3600206
1970,1819164,1918636,3737800
1971,1736774,1826774,3563548
1972,1592347,1673888,3266235
1973,1533102,1613023,3146125


### Task 2 — Multiple Aggregations

Create a pivot table by `year` and `gender` that calculates:

1. total births
2. average daily births
3. maximum daily births
4. minimum daily births

Use multiple aggregation functions in one pivot table.


In [ ]:
births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc=["sum", "mean", "max", "min"]
).head()

sum                  mean                max       min    
gender        F        M            F            M     F     M   F   M
year                                                                  
1969    1753634  1846572  4566.755208  4808.781250  5988  6244  20  16
1970    1819164  1918636  4737.406250  4996.447917  6204  6480   8   6
1971    1736774  1826774  4558.461942  4769.644909  5548  5924   2   2
1972    1592347  1673888  4157.563969  4381.905759  5038  5296   2   2
1973    1533102  1613023  4034.478947  4222.573298  4878  5211   2   2

### Task 3 — Year × Month Matrix

Create a pivot table where:
- rows = year
- columns = month
- values = total births
- aggregation = sum

Then identify the month with the highest total births across all years.


In [ ]:
monthly_births = births.pivot_table(
    values="births",
    index="year",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

monthly_births.head()

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
1969,293940,270786,296550,282638,289124,291610,318356,321034,312620,311972,297054,314522
1970,302278,281488,307448,287090,298140,303378,330452,331326,332496,324422,309604,329678
1971,312826,284052,308312,286304,284926,285518,305534,313122,311534,303248,281762,286410
1972,276544,262862,273645,255075,270279,260229,278268,288570,283076,275903,264669,277115
1973,266483,241692,269506,248082,255267,254799,275934,282189,271605,266416,252780,261372


In [66]:
monthly_totals = monthly_births.sum(axis=0)
highest_month = monthly_totals.idxmax()
highest_value = monthly_totals.max()

highest_month, highest_value

(np.int64(8), np.int64(13528007))

In [ ]:
births.pivot_table(values='births', index='year', columns='gender', aggfunc='sum', sort=True).head()

gender,F,M
year,,
1969,1753634,1846572
1970,1819164,1918636
1971,1736774,1826774
1972,1592347,1673888
1973,1533102,1613023


### Task 4 — Gender Share by Year

Create a year × gender pivot table containing total births.

Then calculate the **percentage of births contributed by each gender within every year**.

Each row should add up to approximately 100%.


In [71]:
gender_year = births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc="sum"
)

gender_year.head()

gender,F,M
year,,
1969,1753634,1846572
1970,1819164,1918636
1971,1736774,1826774
1972,1592347,1673888
1973,1533102,1613023


In [75]:
gender_share = gender_year[['F','M']].div(other=gender_year.sum(axis=1), axis=0) * 100
gender_share.head()

gender,F,M
year,,
1969,48.709268,51.290732
1970,48.669378,51.330622
1971,48.737214,51.262786
1972,48.751759,51.248241
1973,48.729850,51.270150


In [ ]:
gender_share.sum(axis=1).head()


In [ ]:
births.pivot_table(values='births',index='year',columns='gender',aggfunc=['sum']).head()

### Task 5 — Find the Year with the Largest Gender Difference

Using a year × gender pivot table:

1. Calculate total births for each gender.
2. Calculate the absolute difference between male and female births.
3. Find the year with the largest difference.


In [ ]:
gender_year = births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc="sum",
    fill_value=0
)

gender_year["gender_difference"] = (
    gender_year["M"] - gender_year["F"]
).abs()

gender_year.loc[gender_year["gender_difference"].idxmax()]


In [ ]:
gender_year.loc[gender_year["gender_difference"].idxmax()]

In [ ]:
gender_year

### Task 6 — Quarter-Level Pivot

Create a new `quarter` column from `month`.

Then create a pivot table with:
- index = year
- columns = quarter
- values = births
- aggregation = sum

Finally determine the quarter with the highest total number of births.


In [ ]:
births["quarter"] = pd.cut(
    births["month"],
    bins=[0, 3, 6, 9, 12],
    labels=["Q1", "Q2", "Q3", "Q4"]
)
births

In [ ]:
quarter_pivot = births.pivot_table(
    values="births",
    index="year",
    columns="quarter",
    aggfunc="sum",
    fill_value=0,
    observed=False
)
quarter_pivot.head()

In [ ]:
quarter_totals = quarter_pivot.sum(axis=0)
quarter_totals

In [ ]:
quarter_totals.idxmax()

### Task 7 — Advanced MultiIndex Pivot

Create a pivot table using:

- index = `year`, `gender`
- columns = `month`
- values = `births`
- aggregation = `sum`

The result should have a MultiIndex on the rows.

Then calculate the total births for every `(year, gender)` combination.


In [ ]:
advanced_pivot = births.pivot_table(
    values="births",
    index=["year", "gender"],
    columns="month",
    aggfunc="sum",
    fill_value=0
)

advanced_pivot.head()


In [ ]:
advanced_pivot["total_births"] = advanced_pivot.sum(axis=1)
advanced_pivot["total_births"].unstack()

### Task 8 — Monthly Gender Comparison

Create a pivot table with:
- index = month
- columns = gender
- values = births
- aggregation = sum

Add a calculated column called `M_to_F_ratio` representing:

`Male births / Female births`

Find the month with the highest ratio.


In [ ]:
monthly_gender = births.pivot_table(
    values="births",
    index="month",
    columns="gender",
    aggfunc="sum",
    fill_value=0
)

monthly_gender["M_to_F_ratio"] = (
    monthly_gender["M"] / monthly_gender["F"]
)

monthly_gender


In [ ]:
monthly_gender.loc[monthly_gender["M_to_F_ratio"].idxmax()]


### Task 9 — Custom Aggregation Function

Create a pivot table by `year` and `gender` using a custom aggregation function that returns:

`max(births) - min(births)`

Name the resulting measure `birth_range`.


In [ ]:
births.head()

In [ ]:
def birth_range(x):
    return x.max() - x.min()

birth_range_pivot = births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc=birth_range
)

birth_range_pivot


### Task 10 — Detect the Highest-Birth Day

Create a pivot table using:
- index = year
- columns = gender
- values = births
- aggregation = max

Then determine the maximum daily birth count for each gender across all years.


In [ ]:
max_daily = births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc="max"
)
max_daily

In [ ]:
max_daily.max()

### Task 11 — Yearly Statistics with Multiple Metrics

Create a pivot table by `year` with the following metrics for `births`:

- sum
- mean
- median
- standard deviation
- minimum
- maximum

Then add a new column outside the pivot result called `range`, calculated as:

`maximum - minimum`


In [ ]:
year_stats = births.pivot_table(
    values="births",
    index="year",
    aggfunc=["sum", "mean", "median", "std", "min", "max"]
)
year_stats.head()

In [ ]:
year_stats["range"] = (
    year_stats[("max", "births")] - year_stats[("min", "births")]
)

year_stats


### Task 12 — Compare the First and Last Year

Build a pivot table with:
- index = year
- columns = gender
- values = total births

Then calculate the percentage change in births for each gender between the first and last year in the dataset.

Formula:

`((last_year - first_year) / first_year) * 100`


In [ ]:
year_gender = births.pivot_table(
    values="births",
    index="year",
    columns="gender",
    aggfunc="sum"
)
year_gender.head()

In [ ]:
first_year = year_gender.iloc[0]
last_year = year_gender.iloc[-1]
first_year,last_year

In [ ]:
percentage_change = ((last_year - first_year) / first_year) * 100
percentage_change.head()


### Task 13 — Birth Volume Categories

Create a new column called `birth_volume` using `pd.qcut()` to divide individual records into 4 equally populated groups:

- Low
- Medium
- High
- Very High

Then create a pivot table showing:

- index = `birth_volume`
- columns = `gender`
- values = `births`
- aggregation = `sum`



In [ ]:
births["birth_volume"] = pd.qcut(
    births["births"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)
births.head()

In [ ]:

volume_pivot = births.pivot_table(
    values="births",
    index="birth_volume",
    columns="gender",
    aggfunc="sum",
    fill_value=0,
    observed=False
)

volume_pivot


### Task 14 — Identify the Most Important Month for Each Gender

Create a month × gender pivot table containing total births.

For each gender, identify:
- the month with the highest births
- the number of births in that month

Return the result as a small DataFrame.


In [ ]:
month_gender = births.pivot_table(
    values="births",
    index="month",
    columns="gender",
    aggfunc="sum"
)
month_gender.head()

In [ ]:
results = []

for gender in month_gender.columns:
    month = month_gender[gender].idxmax()
    value = month_gender[gender].max()
    results.append({
        "gender": gender,
        "month": month,
        "births": value
    })

pd.DataFrame(results)


### Task 15 — Expert Challenge: Year × Quarter × Gender

Create a three-dimensional analytical pivot table:

- index = `year`
- columns = `quarter`, `gender`
- values = `births`
- aggregation = `sum`

Then:

1. Calculate the total births for each year.
2. Find the year with the highest total births.
3. Find the `(quarter, gender)` combination with the highest overall number of births.


In [ ]:
births.head()

In [ ]:
expert_pivot = births.pivot_table(
    values="births",
    index="year",
    columns=["quarter", "gender"],
    aggfunc="sum",
    fill_value=0,
    observed=False
)

expert_pivot.head()


In [ ]:
year_totals = expert_pivot.sum(axis=1)
year_totals.head()

In [ ]:
highest_year = year_totals.idxmax()
highest_year

In [ ]:
overall_combination = expert_pivot.sum(axis=0)
overall_combination


In [ ]:
highest_combination = overall_combination.idxmax()
highest_combination

In [ ]:
highest_combination_value = overall_combination.max()
highest_combination_value

In [ ]:

highest_combination_value = overall_combination.max()

print("Year with highest total births:", highest_year)
print("Total births:", year_totals.max())
print("Highest quarter/gender combination:", highest_combination)
print("Births:", highest_combination_value)
